# Clasificación few-shot de **conjuntos de documentos**: tres clasificadores geométricos

**Unidad de evaluación:** cada consulta es un conjunto de `M_QUERY_DOCS` documentos de la misma clase, agrupados usando las etiquetas *solo al preparar el episodio*. `K_VALUES` determina los documentos etiquetados por clase.

- **Shimomoto TF-MSM:** mantiene su representación original basada en **vectores por palabra, frecuencia de términos y PCA sin centrar**. Los K documentos de entrenamiento de una clase aportan sus términos al subespacio de clase. Cada uno de los M documentos de consulta se compara por separado contra cada clase mediante ángulos canónicos; se promedian las **similitudes**, no los embeddings, para obtener una predicción del conjunto. Para Transformers, los vectores por término son contextualizados y sus ocurrencias se promedian: esta es una extensión explícita de Word2Vec, no la implementación original exacta.
- **Sogi MCM:** utiliza **un embedding por documento**, cono por conjunto y generadores comprimidos con NMF.
- **Exceso angular:** compara conos de documentos; usa la versión de gradiente proyectado, no cutting-plane.

No hay fuga entre entrenamiento y prueba. Word2Vec Google News reproduce la familia de características de Shimomoto; Transformers se incluyen como comparación moderna. Se limitan términos por hiperparámetros para controlar recursos.

Referencias: https://arxiv.org/pdf/1806.03125 ; https://arxiv.org/pdf/1805.12467 ; https://arxiv.org/abs/2511.10483


## 0. Instalar dependencias

La instalación puede omitirse al ejecutar localmente si ya estan instaladas.

In [1]:
import sys, subprocess, os
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '-U',
        'datasets>=3.0,<5', 'transformers>=4.45,<5', 'gensim>=4.3',
        'huggingface_hub>=0.28', 'scikit-learn>=1.4', 'scipy>=1.12',
        'pandas>=2.0', 'matplotlib>=3.8', 'tqdm>=4.66'])


## 1. Configuración

`M_QUERY_DOCS = 5`: número de documentos por **conjunto de consulta** (por defecto; modificable). `N_QUERY`: número de conjuntos de consulta por clase y episodio. `K_VALUES`: documentos de soporte por clase. Modificar M no cambia K. En PILOTO=True se hace una prueba corta.


In [2]:
import gc
import json
import hashlib
import re
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import svd
from scipy.optimize import nnls
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

SEED = 2026
PILOTO = False
K_final = 20
K_VALUES = list(range(1, K_final + 1)) if not PILOTO else [1, 2, 3]
K_MAX = max(K_VALUES)
N_WAY = 5
N_EPISODES = 3 if not PILOTO else 1
N_QUERY = 10                        # conjuntos de consulta por clase y episodio
M_QUERY_DOCS = 5                 # documentos/generadores por consulta; independiente de K
DOC_MAX_WORDS = 48                # igual para todos los embeddings
TRANSFORMER_MAX_PIECES = 96       # limite de subtokens
BATCH_SIZE = 16
MAX_CLASS_TERMS = 512             # palabras unicas mas frecuentes en el soporte TF-MSM
MAX_QUERY_TERMS = 48              # palabras unicas por documento de consulta TF-MSM
SUBSPACE_RANK = 8                  # dimension de bases TF-MSM
CONE_RANK = 4                      # numero de generatrices NMF para MCM
N_ANGLES = 3                      # distinto de K-shot
ALS_STEPS = 8
ALS_TOL = 1e-4
DATASET_SEED_CLASSES_FACTOR = 3   # maximo de clases disponibles por dataset
MAX_STREAM_ROWS = 1_000_000
OUTPUT_DIR = Path('/content/resultados_tres_metodos_ruido' if 'google.colab' in sys.modules
                  else './resultados_tres_metodos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS = (
    ['sentence-transformers/all-MiniLM-L6-v2'] if PILOTO else
    ['word2vec-google-news-300',
     'sentence-transformers/all-MiniLM-L6-v2',
     'BAAI/bge-small-en-v1.5']
)

@dataclass(frozen=True)
class DatasetSpec:
    name: str
    repo: str
    n_classes: int
    text_column: str = 'text'
    label_column: str = 'label'
    config: str | None = None
    train_split: str = 'train'
    test_split: str = 'test'

DATASETS = [
    DatasetSpec('AG_News', 'fancyzhx/ag_news', 4),
    DatasetSpec('DBpedia_14', 'fancyzhx/dbpedia_14', 14, text_column='content'),
    DatasetSpec('BBC_News', 'SetFit/bbc-news', 5),
    # DatasetSpec('TREC_6', 'SetFit/TREC-QC', 6, label_column='label_coarse'),
    # DatasetSpec('Banking77', 'PolyAI/banking77', 77),
    DatasetSpec('20_Newsgroups', 'SetFit/20_newsgroups', 20),
    DatasetSpec('Emotion_6', 'dair-ai/emotion', 6, config='split'),
    # DatasetSpec('Yelp_5', 'Yelp/yelp_review_full', 5),
    # DatasetSpec('MASSIVE_60', 'SetFit/amazon_massive_intent_en-US', 60),
    DatasetSpec('SST_5', 'SetFit/sst5', 5),
]
if PILOTO:
    DATASETS = [next(s for s in DATASETS if s.name == 'BBC_News')]

print(f'Datasets: {len(DATASETS)} | Embeddings: {len(EMBEDDINGS)} | K: {K_VALUES}')
print(f'Episodios={N_EPISODES}, conjuntos/clase={N_QUERY}, documentos/conjunto={M_QUERY_DOCS}, angulos={N_ANGLES}')


Datasets: 6 | Embeddings: 3 | K: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Episodios=3, conjuntos/clase=10, documentos/conjunto=5, angulos=3


## 2. Episodios sin fuga: conjuntos homogéneos de consulta

Cada clase recibe K documentos de `train`, con conjuntos anidados al aumentar K. Cada consulta es un **grupo de `M_QUERY_DOCS` documentos distintos del split test y de una misma clase**. La etiqueta se utiliza únicamente para construir los grupos y evaluar, nunca para escoger la clase predicha. `N_QUERY` cuenta **grupos**, no documentos. Todos los métodos y embeddings reciben idénticos grupos de consulta.


In [3]:
from datasets import load_dataset

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
STOP = set(ENGLISH_STOP_WORDS)

def words_of(text, max_words=DOC_MAX_WORDS):
    # Solo se estudian datasets ingleses para utilizar Google News Word2Vec.
    toks = [w.lower() for w in WORD_RE.findall(str(text))]
    toks = [w for w in toks if len(w) > 1 and w not in STOP]
    return toks[:max_words]

def stable_seed(*parts):
    h = hashlib.sha256('::'.join(map(str, parts)).encode('utf8')).digest()
    return int.from_bytes(h[:4], 'little')

def stream_split(spec, split, seed):
    kwargs = {'path': spec.repo, 'split': split, 'streaming': True}
    if spec.config is not None:
        kwargs['name'] = spec.config
    ds = load_dataset(**kwargs)
    return ds.shuffle(seed=seed, buffer_size=1024)

def gather_by_class(spec, split, selected_classes, quota, seed):
    result = {int(c): [] for c in selected_classes}
    for row_i, ex in enumerate(stream_split(spec, split, seed)):
        if row_i >= MAX_STREAM_ROWS:
            break
        c = int(ex[spec.label_column])
        if c not in result or len(result[c]) >= quota:
            continue
        text = str(ex[spec.text_column] or '').strip()
        if len(words_of(text)) < 2:
            continue
        # Evita duplicados exactos dentro de un mismo grupo.
        result[c].append({'id': f'{split}:{c}:{len(result[c])}',
                          'label': c, 'text': text})
        if all(len(v) >= quota for v in result.values()):
            break
    return result

def make_pools(spec):
    cache = OUTPUT_DIR / f'pool_{spec.name}.json'
    if cache.exists():
        with open(cache, encoding='utf8') as fh:
            pool = json.load(fh)
        if (pool['version'] == 3 and pool['kmax'] >= K_MAX and
            pool['n_episodes'] >= N_EPISODES and pool['n_query'] >= N_QUERY and pool.get('m_query_docs') == M_QUERY_DOCS):
            return pool

    rng = np.random.default_rng(stable_seed(SEED, spec.name, 'classes'))
    nc = min(spec.n_classes, max(N_WAY, N_WAY * DATASET_SEED_CLASSES_FACTOR))
    classes = rng.choice(spec.n_classes, size=nc, replace=False).tolist()
    train_quota = K_MAX + 5
    test_quota = N_EPISODES * N_QUERY * M_QUERY_DOCS + 3
    train = gather_by_class(spec, spec.train_split, classes, train_quota,
                            stable_seed(SEED, spec.name, 'train'))
    test = gather_by_class(spec, spec.test_split, classes, test_quota,
                           stable_seed(SEED, spec.name, 'test'))
    eligible = [c for c in classes if len(train[int(c)]) >= K_MAX and
                len(test[int(c)]) >= N_QUERY * M_QUERY_DOCS]
    if len(eligible) < min(N_WAY, spec.n_classes):
        sizes = {c: (len(train[int(c)]), len(test[int(c)])) for c in classes}
        raise RuntimeError(f'Clases insuficientes en {spec.name}: {sizes}. '
                           'Revisar MAX_STREAM_ROWS/splits o elegir otras clases.')
    pool = {'version': 3, 'm_query_docs': M_QUERY_DOCS, 'kmax': K_MAX, 'n_episodes': N_EPISODES,
            'n_query': N_QUERY, 'classes': eligible,
            'train': {str(c): train[int(c)] for c in eligible},
            'test': {str(c): test[int(c)] for c in eligible}}
    with open(cache, 'w', encoding='utf8') as fh:
        json.dump(pool, fh, ensure_ascii=False)
    print(f' {spec.name}: {len(eligible)} clases aptas, train/test independientes')
    return pool

def create_episodes(spec, pool):
    result = []
    for ep in range(N_EPISODES):
        rng = np.random.default_rng(stable_seed(SEED, spec.name, ep))
        classes = sorted(int(x) for x in rng.choice(
            pool['classes'], size=min(N_WAY, spec.n_classes), replace=False))
        support = {}
        queries = []
        for c in classes:
            s = pool['train'][str(c)]
            q = pool['test'][str(c)]
            i_s = rng.choice(len(s), size=K_MAX, replace=False)
            i_q = rng.choice(len(q), size=N_QUERY * M_QUERY_DOCS, replace=False)
            support[c] = [s[int(i)] for i in i_s]
            selected = [q[int(i)] for i in i_q]
            for j in range(N_QUERY):
                group = selected[j*M_QUERY_DOCS:(j+1)*M_QUERY_DOCS]
                queries.append({'id': f'{spec.name}:ep{ep}:class{c}:set{j}',
                                'label': c, 'docs': group})
        rng.shuffle(queries)
        result.append({'episode': ep, 'classes': classes,
                       'support': support, 'queries': queries})
    return result


## 3. Extracción dual de características

Se guardan **tokens individuales con sus embeddings** para Shimomoto TF-MSM y, en paralelo, **un embedding global por documento** para Sogi y exceso angular. Word2Vec usa Google News; MiniLM/BGE usan vectores contextualizados por palabra. Los Transformers se agregan por media de vectores de tokens para formar una representación documental únicamente en los métodos basados en conos de documentos.


In [4]:
import torch
from transformers import AutoModel, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, '| CUDA:', torch.cuda.is_available())

class GoogleNewsEncoder:
    def __init__(self):
        import gensim.downloader as api
        print('Descargando/cargando Google News Word2Vec; requiere RAM y disco.')
        self.model = api.load('word2vec-google-news-300')
        self.dim = int(self.model.vector_size)
        self.name = 'word2vec-google-news-300'

    def encode(self, texts):
        out = []
        for text in texts:
            words = words_of(text)
            keep, values = [], []
            for w in words:
                key = next((u for u in [w, w.capitalize(), w.upper()]
                            if u in self.model.key_to_index), None)
                if key is not None:
                    keep.append(w)
                    values.append(np.asarray(self.model[key], dtype=np.float32))
            mat = np.asarray(values, dtype=np.float32).reshape(-1, self.dim)
            out.append((keep, normalize_rows(mat)))
        return out

class ContextualTokenEncoder:
    def __init__(self, name):
        self.name = name
        self.tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True)
        if not self.tokenizer.is_fast:
            raise ValueError(f'{name} requiere tokenizer fast con word_ids().')
        self.model = AutoModel.from_pretrained(name).to(DEVICE).eval()
        self.dim = self.model.config.hidden_size

    def encode(self, texts):
        outputs = []
        tokens = [words_of(t) for t in texts]
        for start in range(0, len(texts), BATCH_SIZE):
            batch_tokens = tokens[start:start + BATCH_SIZE]
            tok = self.tokenizer(batch_tokens, is_split_into_words=True,
                                 padding=True, truncation=True,
                                 max_length=TRANSFORMER_MAX_PIECES,
                                 return_tensors='pt')
            ids_per_text = [tok.word_ids(batch_index=j)
                            for j in range(len(batch_tokens))]
            inputs = {k: v.to(DEVICE) for k, v in tok.items()}
            with torch.inference_mode():
                hidden = self.model(**inputs).last_hidden_state.float().cpu().numpy()
            for j, raw_words in enumerate(batch_tokens):
                groups = defaultdict(list)
                for token_idx, word_idx in enumerate(ids_per_text[j]):
                    if word_idx is not None:
                        groups[int(word_idx)].append(hidden[j, token_idx])
                ix = sorted(groups)
                keys = [raw_words[i] for i in ix]
                mat = (np.stack([np.mean(groups[i], axis=0) for i in ix])
                       if ix else np.zeros((0, self.dim), dtype=np.float32))
                outputs.append((keys, normalize_rows(mat.astype(np.float32))))
            del tok, inputs, hidden
        return outputs

def normalize_rows(mat):
    if mat.size == 0:
        return mat
    den = np.maximum(np.linalg.norm(mat, axis=1, keepdims=True), 1e-12)
    return (mat / den).astype(np.float32)

def instantiate_encoder(name):
    return GoogleNewsEncoder() if name == 'word2vec-google-news-300' else ContextualTokenEncoder(name)

def normalize_document_embedding(token_matrix, dim):
    if token_matrix.size == 0:
        return np.zeros(dim, dtype=np.float32)
    v = np.mean(token_matrix, axis=0)
    norm = np.linalg.norm(v)
    return (v / norm).astype(np.float32) if norm > 1e-12 else np.zeros(dim, dtype=np.float32)

def encode_pool(encoder, pool):
    records = []
    for split in ('train', 'test'):
        for cls_records in pool[split].values():
            records.extend(cls_records)
    embeddings = {}
    for start in tqdm(range(0, len(records), 128), desc='Embeddings', leave=False):
        chunk = records[start:start + 128]
        vectors = encoder.encode([ex['text'] for ex in chunk])
        for ex, (keys, mat) in zip(chunk, vectors):
            embeddings[ex['id']] = {'tokens': (keys, mat), 'document': normalize_document_embedding(mat, encoder.dim)}
    cover = sum(np.linalg.norm(embeddings[ex['id']]['document']) > 1e-10 for ex in records) / max(1, len(records))
    print(f' {encoder.name}: cobertura de documentos = {cover:.1%}')
    return embeddings


Dispositivo: cpu | CUDA: False


## 4. Shimomoto: TF-MSM basado en palabras (sin promediar documentos)

Para cada clase, se agregan las ocurrencias de palabras de los K textos de soporte; cada término tiene su embedding y frecuencia. Se construye el subespacio con SVD sin centrar sobre `X * sqrt(TF)`. Para consulta, **cada documento genera un subespacio propio de palabras** y la similitud con cada clase se calcula usando cosenos cuadrados de ángulos canónicos. Solo al final se promedian las similitudes de los M documentos de consulta para obtener una decisión por conjunto; no se promedian embeddings para Shimomoto.

La implementación original usa Word2Vec; en Transformers, se agregan ocurrencias contextualizadas por término. `MAX_CLASS_TERMS`, `MAX_QUERY_TERMS`, `SUBSPACE_RANK` y `N_ANGLES` son hiperparámetros experimentales y la selección por frecuencia de términos constituye una restricción de recursos.


In [5]:
def aggregate_tokens(docs, embeddings, max_terms):
    vec_sum = {}
    freq = Counter()
    for ex in docs:
        keys, mat = embeddings[ex['id']]['tokens']
        for w, x in zip(keys, mat):
            if w not in vec_sum:
                vec_sum[w] = np.zeros_like(x)
            vec_sum[w] += x
            freq[w] += 1
    if not freq:
        dim = next(iter(embeddings.values()))['document'].shape[0]
        return (np.zeros((0, dim), dtype=np.float32),
                np.zeros(0, dtype=np.float32))
    ranked = sorted(freq, key=lambda w: (-freq[w], w))[:max_terms]
    X = np.stack([vec_sum[w] / freq[w] for w in ranked]).astype(np.float32)
    X = normalize_rows(X)
    tf = np.array([freq[w] for w in ranked], dtype=np.float32)
    return X, tf

def tf_subspace(X, tf, rank=SUBSPACE_RANK):
    # X: n_terms x dim, X_tf^T: dim x n_terms.
    if len(X) == 0:
        return np.zeros((X.shape[1], 0), dtype=np.float32)
    A = (X * np.sqrt(tf)[:, None]).T
    U, s, _ = svd(A, full_matrices=False, check_finite=False)
    valid = int(np.sum(s > max(1e-7, s[0] * 1e-5))) if len(s) else 0
    return U[:, :min(rank, valid)].astype(np.float32)

def tf_msm_similarity(U, V, n_angles=N_ANGLES):
    if U.shape[1] == 0 or V.shape[1] == 0:
        return 0.0
    cosines = svd(U.T @ V, compute_uv=False, check_finite=False)
    t = min(n_angles, len(cosines))
    return float(np.mean(np.clip(cosines[:t], 0, 1) ** 2))


## 5. Sogi MCM a nivel documento

Una columna es un embedding de un documento. Se aplica sign-splitting (solo para NMF sobre datos no negativos) y NMF para obtener generadores de cono; la similitud MCM promedia cosenos cuadrados de ángulos sucesivos. Esta transformación es una **adaptación** para embeddings firmados, no la extracción original de CNN-ReLU.


In [6]:
def sign_split(X):
    return np.concatenate([np.maximum(X, 0), np.maximum(-X, 0)], axis=1)

def cone_basis(X, tf, rank=CONE_RANK):
    if len(X) == 0:
        return np.zeros((2 * X.shape[1], 0), dtype=np.float32)
    A = (sign_split(X) * np.sqrt(tf)[:, None]).astype(np.float64)
    n_comp = min(rank, A.shape[0], A.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        nmf = NMF(n_components=n_comp, init='nndsvda', random_state=0,
                  solver='cd', max_iter=70, tol=3e-3)
        nmf.fit(A)
    H = nmf.components_.T
    norms = np.linalg.norm(H, axis=0)
    valid = norms > 1e-8
    if not np.any(valid):
        return np.zeros((A.shape[1], 0), dtype=np.float32)
    return (H[:, valid] / norms[valid]).astype(np.float64)

def project_cone(B, y):
    # B: dim x numero de generatrices; restringir coeficientes a >=0
    if B.shape[1] == 0:
        return np.zeros_like(y)
    coef, _ = nnls(B, y)
    return B @ coef

def orthogonal_complement_basis(B, Q):
    # La imagen ortogonal de un cono sigue siendo un cono generado por
    # las proyecciones de sus generatrices (aunque tengan signos).
    return B - Q @ (Q.T @ B) if Q.shape[1] else B

def mcm_similarity(B1, B2, n_angles=N_ANGLES):
    if not B1.shape[1] or not B2.shape[1]:
        return 0.0
    Q = np.zeros((B1.shape[0], 0), dtype=np.float64)
    cos2s = []
    for _ in range(min(n_angles, B1.shape[1], B2.shape[1])):
        C1 = orthogonal_complement_basis(B1, Q)
        C2 = orthogonal_complement_basis(B2, Q)
        y = C1.sum(axis=1) + C2.sum(axis=1)
        y_norm = np.linalg.norm(y)
        if y_norm < 1e-9:
            y = C1[:, 0] + C2[:, 0]
            y_norm = np.linalg.norm(y)
            if y_norm < 1e-9:
                break
        y = y / y_norm
        p = q = None
        for it in range(ALS_STEPS):
            p = project_cone(C1, y)
            q = project_cone(C2, y)
            np_, nq_ = np.linalg.norm(p), np.linalg.norm(q)
            if min(np_, nq_) < 1e-9:
                break
            p, q = p / np_, q / nq_
            new_y = (p + q) / 2.
            if np.linalg.norm(new_y - y) < ALS_TOL:
                break
            y = new_y
        if p is None or q is None or min(np.linalg.norm(p), np.linalg.norm(q)) < 1e-8:
            break
        # Coseno cuadrado, pues el articulo promedia correlaciones cuadradas.
        cos2s.append(float(np.clip(p @ q, -1, 1) ** 2))
        # Gram-Schmidt/SVD para acumular span de los pares anteriores.
        candidates = np.column_stack([Q, p, q])
        U, s, _ = svd(candidates, full_matrices=False, check_finite=False)
        effective = int(np.sum(s > max(1e-8, s[0] * 1e-6)))
        Q = U[:, :effective]
        if Q.shape[1] >= B1.shape[0]:
            break
    return float(np.mean(cos2s)) if cos2s else 0.0


## 5b. Pruebas matematicas de consistencia

Un subespacio/cono consigo mismo debe tener similitud 1 (un angulo).
Dos rayos ortogonales deben tener similitud 0.

## 6. Tercer clasificador: exceso angular max-min de conos convexos

Sea Pq = cone(G) el cono de la consulta, y Qc = cone(Hc) el cono de la clase c.

El **exceso angular dirigido** es

$$\Theta(P_q,Q_c) = \max_{u\in P_q\cap S}\min_{v\in Q_c\cap S}\arccos(u^\top v)
=\arccos\!\left(\min_{u\in P_q\cap S}F_{Q_c}(u)\right),$$
con $F_Q(u)=\max_{v\in Q\cap S}\langle u,v\rangle$. Se predice $\hat c=\arg\min_c\Theta(P_q,Q_c)$.

**Opcion simetrica** del articulo: $\mathrm{Dis}_r(P_q,Q_c) = 2 \left\| (\sin(\Theta(P_q,Q_c)/2),\sin(\Theta(Q_c,P_q)/2)) \right\|_r$ (configurar `EXCESS_DIRECTION='symmetric'`, `EXCESS_R=2`).

**Diferencia frente al articulo:** este codigo utiliza un algoritmo de *gradiente proyectado* con arranques multiples, no el *cutting-plane* de su Algoritmo 5.1; por tanto, estima una solucion **local** y no garantiza el max-min global. La funcion soporte incluye el caso polar, en que puede ser negativa; alli `norm(proj_Q(u))` por si sola seria incorrecta.

**Representacion:** `sogi_nmf` compara los dos criterios angulares sobre las **mismas matrices generadoras de NMF**. `raw_docs` usa vectores de palabras originales (con signo) para generar los conos, como analisis alternativo; en Colab puede ser considerablemente mas lento. Los pesos TF solo intervienen en NMF cuando se usa `sogi_nmf`.


In [7]:
# Tercer clasificador: exceso angular max-min (adaptacion robusta del codigo del usuario)
# G y H deben ser (dim_embedding, num_generadores), columnas normalizadas.
# NNLS restringe los COEFICIENTES a ser no negativos; los vectores pueden tener signo.
from scipy.optimize import nnls, lsq_linear

EXCESS_DIRECTION = 'symmetric'#'directed'      # 'directed' (solicitado) o 'symmetric'
EXCESS_CONE_SOURCE = 'sogi_nmf'   # 'sogi_nmf' (misma geometria que Sogi) o 'raw_docs'
EXCESS_R = 2                      # norma r para symmetric: 1, 2, np.inf
EXCESS_MAX_ITER = 25
EXCESS_N_STARTS = 3
EXCESS_STEP = 0.5
EXCESS_TOL = 1e-5
EXCESS_METHOD_NAME = ('Exceso angular' if EXCESS_DIRECTION == 'directed'
                      else f'Distancia Simétrica')


def normalize_generators(A, eps=1e-12):
    A = np.asarray(A, dtype=np.float64)
    if A.ndim != 2:
        raise ValueError('Se espera una matriz 2D de generadores (dimension x n_generadores).')
    if not np.all(np.isfinite(A)):
        raise ValueError('Hay NaN o valores infinitos en los generadores.')
    norms = np.linalg.norm(A, axis=0)
    good = norms > eps
    return A[:, good] / norms[good]


def proj_cone_exceso(G, u):
    if G.shape[1] == 0:
        raise ValueError('El cono no puede estar generado por cero vectores.')
    try:
        coef, _ = nnls(G, u, maxiter=max(3 * G.shape[1], 100))
    except (RuntimeError, ValueError):
        # Alternativa robusta para conos numericamente casi degenerados.
        coef = lsq_linear(G, u, bounds=(0, np.inf), max_iter=200).x
    return G @ coef


def soporte_esferico(H, u, tol=1e-12):
    """Devuelve F_Q(u), un maximizador v en Q cap S, y estado polar.

    F_Q(u)=max_{v in Q cap S} <u,v> puede ser NEGATIVO si u pertenece
    al interior del cono polar. No usar solo ||Proj_Q(u)|| en ese caso.
    (Proposicion 5.1 del articulo de Oliveira, Sessa y Sossa).
    """
    dots = H.T @ u
    j = int(np.argmax(dots))
    if dots[j] <= tol:
        # Para u en el polar, el maximo lo alcanza un generador unitario.
        return float(dots[j]), H[:, j].copy(), True
    proj = proj_cone_exceso(H, u)
    norm_proj = float(np.linalg.norm(proj))
    if norm_proj <= tol:
        return float(dots[j]), H[:, j].copy(), True
    v = proj / norm_proj
    return float(np.clip(u @ v, -1.0, 1.0)), v, False


def linesearch_exceso(G, H, u, f_u, grad, t0=EXCESS_STEP, decay=0.5,
                       max_backtracks=16, tol=1e-11):
    """Backtracking sobre el punto re-proyectado y normalizado.

    Una busqueda de Armijo en el paso euclideo NO garantiza descenso una
    vez proyectado sobre P y renormalizado; se evalua el punto factible.
    """
    t = t0
    for _ in range(max_backtracks):
        p = proj_cone_exceso(G, u - t * grad)
        norm_p = np.linalg.norm(p)
        if norm_p > 1e-12:
            u_new = p / norm_p
            f_new, v_new, polar = soporte_esferico(H, u_new)
            if f_new < f_u - tol:
                return u_new, f_new, v_new, polar, t
            if np.linalg.norm(u_new - u) <= EXCESS_TOL:
                break
        t *= decay
    return None


def exceso_gradiente_proyectado(G, H, u0=None, t=EXCESS_STEP,
                                  max_iter=EXCESS_MAX_ITER, eps=EXCESS_TOL,
                                  n_starts=EXCESS_N_STARTS):
    """Estimacion LOCAL del angulo max-min dirigido Theta(P,Q).

    P=cone(G), Q=cone(H). Resuelve aproximadamente
      cos Theta(P,Q) = min_{u en P cap S} F_Q(u).
    Usa la heuristica por generadores del Algoritmo 5.2 como warm start,
    y descenso proyectado propuesto por el usuario; NO implementa el
    algoritmo cutting-plane 5.1 del articulo ni asegura optimo global.
    """
    tic = time.perf_counter()
    G, H = normalize_generators(G), normalize_generators(H)
    if G.shape[0] != H.shape[0] or not G.shape[1] or not H.shape[1]:
        raise ValueError('Conos invalidos: ambos deben tener generadores no nulos en R^d.')
    starts = [G[:, j].copy() for j in range(G.shape[1])]
    sum_g = G.sum(axis=1)
    if np.linalg.norm(sum_g) > 1e-12:
        starts.append(sum_g / np.linalg.norm(sum_g))
    if u0 is not None:
        u0 = proj_cone_exceso(G, np.asarray(u0, dtype=np.float64))
        if np.linalg.norm(u0) > 1e-12:
            starts.append(u0 / np.linalg.norm(u0))

    scored = [(soporte_esferico(H, u)[0], idx, u)
              for idx, u in enumerate(starts)]
    scored.sort(key=lambda z: (z[0], z[1]))
    best = None
    # Se prefiere la mejor generatriz/mezcla por disimilitud.
    for _, _, initial in scored[:max(1, int(n_starts))]:
        u = initial.copy()
        f, v, is_polar = soporte_esferico(H, u)
        converged = False
        changes = []
        k = 0
        for k in range(1, int(max_iter) + 1):
            # v es gradiente fuera del polar y subgradiente dentro del polar.
            result = linesearch_exceso(G, H, u, f, v, t0=t)
            if result is None:
                converged = True  # estacionariedad numerica, no globalidad
                break
            new_u, new_f, new_v, new_polar, step = result
            delta = float(np.linalg.norm(new_u - u))
            changes.append(delta)
            improvement = f - new_f
            u, f, v, is_polar = new_u, new_f, new_v, new_polar
            if delta < eps or improvement < eps * 0.01:
                converged = True
                break
        state = {'u_opt': u, 'v_opt': v, 'coseno': float(np.clip(f, -1., 1.)),
                 'angulo': float(np.arccos(np.clip(f, -1., 1.))),
                 'k': k, 'converged': converged,
                 'polar': bool(is_polar), 'valores_normas': changes}
        if best is None or state['coseno'] < best['coseno']:
            best = state
    best['tiempo_final'] = time.perf_counter() - tic
    best['n_starts'] = min(len(scored), max(1, int(n_starts)))
    return best


def distancia_conos_exceso(G, H, direction=EXCESS_DIRECTION, r=EXCESS_R):
    """Distancia angular dirigida Theta(P,Q) o distancia simetrica Dis_r.

    'directed': Theta(P,Q), la regla solicitada argmin_c Theta(Pq,Qc).
    'symmetric': 2 ||(sin(Theta(P,Q)/2), sin(Theta(Q,P)/2))||_r,
                 regla usada en seccion 6.1.1 del articulo.
    """
    fw = exceso_gradiente_proyectado(G, H)
    if direction == 'directed':
        return fw['angulo'], {'theta_PQ': fw['angulo'], 'theta_QP': np.nan,
                              'cos_PQ': fw['coseno'], 'cos_QP': np.nan,
                              'runtime_s': fw['tiempo_final'],
                              'polar_PQ': fw['polar'], 'polar_QP': None,
                              'conv_PQ': fw['converged'], 'conv_QP': None}
    if direction != 'symmetric':
        raise ValueError('direction debe ser directed o symmetric.')
    rev = exceso_gradiente_proyectado(H, G)
    v = np.array([np.sin(fw['angulo'] / 2), np.sin(rev['angulo'] / 2)])
    d = 2.0 * float(np.linalg.norm(v, ord=r))
    return d, {'theta_PQ': fw['angulo'], 'theta_QP': rev['angulo'],
               'cos_PQ': fw['coseno'], 'cos_QP': rev['coseno'],
               'runtime_s': fw['tiempo_final'] + rev['tiempo_final'],
               'polar_PQ': fw['polar'], 'polar_QP': rev['polar'],
               'conv_PQ': fw['converged'], 'conv_QP': rev['converged']}


def cone_generators_for_excess(representation):
    # Tupla (U_subespacio, B_MCM, T_docs).
    return representation[1] if EXCESS_CONE_SOURCE == 'sogi_nmf' else representation[2]


In [8]:
# Clasificador de exceso angular con gradiente proyectado y ruido gaussiano.
# Conos: matrices G,H de forma (dimension_embedding, n_generadores).
# Shimomoto TF-MSM y Sogi MCM no se modifican.
from scipy.optimize import nnls, lsq_linear
import numpy as np
import time

EXCESS_DIRECTION = 'directed'        # 'directed' o 'symmetric'
EXCESS_CONE_SOURCE = 'raw_docs'      # UN embedding por documento (recomendado)
# Alternativa: 'sogi_nmf' -> reutilizar las bases NMF de Sogi.
EXCESS_R = 2                        # orden de la norma si symmetric
EXCESS_MAX_ITER = 60                # 5000 como en el prototipo seria muy costoso para un barrido completo
EXCESS_N_STARTS = 1                 # 1 reproduce el arranque suma(G); >1 robustez adicional
EXCESS_ETA = 0.1
EXCESS_SIGMA0 = 0.05
EXCESS_DECAY_RUIDO = 0.001
EXCESS_VENTANA = 30
EXCESS_FACTOR_ESCAPE = 5.0
EXCESS_EPS = 1e-6
EXCESS_TOL_POLAR = 1e-8
EXCESS_METHOD_NAME = ('Exceso' if EXCESS_DIRECTION == 'directed'
                      else f'Simétrico')


def normalize_generators(A, eps=1e-12):
    A = np.asarray(A, dtype=np.float64)
    if A.ndim != 2 or not np.all(np.isfinite(A)):
        raise ValueError('Matriz de generadores 2D y finita requerida.')
    norms = np.linalg.norm(A, axis=0)
    good = norms > eps
    return A[:, good] / norms[good]


def es_polar(u, H, tol=1e-8):
    return bool(np.all(H.T @ u <= tol))


def proj_cone(G, u):
    """Proyeccion euclidiana sobre cone(G) por NNLS; coeficientes >= 0."""
    if G.ndim != 2 or G.shape[1] == 0:
        raise ValueError('G debe tener al menos un generador.')
    try:
        x, _ = nnls(G, u, maxiter=max(3*G.shape[1], 100))
    except (RuntimeError, ValueError):
        x = lsq_linear(G, u, bounds=(0.0, np.inf), max_iter=200).x
    return G @ x


def _soporte_y_direccion(H, u, proy=None, tol_polar=1e-8):
    """max_{v in cone(H) cap S} <u,v> y direccion que lo realiza.

    En la region polar, ||Proj_Q(u)|| = 0 y no permite recuperar el angulo.
    Si H tiene columnas unitarias, el mejor v polar es una columna de H.
    """
    dots = H.T @ u
    j = int(np.argmax(dots))
    if dots[j] <= tol_polar:
        return float(np.clip(dots[j], -1., 1.)), H[:, j].copy(), True
    if proy is None:
        proy = proj_cone(H, u)
    nv = float(np.linalg.norm(proy))
    if nv < 1e-12:
        return float(np.clip(dots[j], -1., 1.)), H[:, j].copy(), True
    return float(np.clip(u @ proy / nv, -1., 1.)), proy/nv, False


def exceso_gradiente_ruido(G, H, u0=None, eta=0.1, sigma0=0.05,
                           decay_ruido=0.001, max_iter=5000, eps=1e-6,
                           ventana=30, factor_escape=5.0, random_state=None,
                           tol_polar=1e-8):
    """Minimiza aproximadamente 0.5 ||Proj_cone(H)(u)||**2 sobre cone(G) cap S.

    Se perturba la ACTUALIZACION, no las columnas de G ni H.
    Es una heuristica estocastica local, no un algoritmo global de max-min.
    """
    inicio = time.perf_counter()
    rng = np.random.default_rng(random_state)
    G = normalize_generators(G)
    H = normalize_generators(H)
    if G.shape[0] != H.shape[0] or not G.shape[1] or not H.shape[1]:
        raise ValueError('Conos no vacios, de igual dimension, con generadores no nulos.')
    if eta <= 0 or sigma0 < 0 or decay_ruido < 0 or ventana < 1 or factor_escape < 1:
        raise ValueError('Configuracion eta/ruido/escape invalida.')
    if max_iter < 1 or eps <= 0 or tol_polar <= 0:
        raise ValueError('max_iter, eps y tol_polar deben ser positivos.')

    n, p = G.shape
    if u0 is None:
        u = G @ np.ones(p)
        if np.linalg.norm(u) < 1e-12:
            u = G[:, 0].copy()  # evita suma nula con generadores opuestos
    else:
        u0 = np.asarray(u0, dtype=float).reshape(-1)
        if u0.shape != (n,) or not np.all(np.isfinite(u0)):
            raise ValueError('u0 debe ser vector finito de longitud G.shape[0].')
        u = proj_cone(G, u0)  # garantiza factibilidad: u en P
        if np.linalg.norm(u) < 1e-12:
            raise ValueError('La proyeccion de u0 sobre P es nula.')
    u = u / np.linalg.norm(u)

    proy = proj_cone(H, u)
    F = 0.5 * float(np.dot(proy, proy))
    cos_act, v_act, polar_act = _soporte_y_direccion(H, u, proy, tol_polar)
    u_best, v_best = u.copy(), v_act.copy()
    F_best, cos_best, polar_best = F, cos_act, polar_act
    historial_objetivo = [F]
    historial_ruido, historial_cambios = [], []
    sin_mejora = 0
    sigma_actual = sigma0
    converged = False
    motivo_parada = 'max_iter alcanzado'
    k_final = 0

    for k in range(1, max_iter + 1):
        k_final = k
        if es_polar(u, H, tol=tol_polar):
            converged = True
            motivo_parada = ('Se encontro direccion polar (F=0; '
                             'Theta no necesariamente optimo).')
            break
        # Gradiente exacto de 1/2 ||Proj_Q(u)||^2.
        grad = proy
        sigma_base = sigma0 / (1.0 + decay_ruido*(k-1))
        if sin_mejora >= ventana:
            sigma_actual = factor_escape * sigma_base
            sin_mejora = 0
        else:
            sigma_actual = sigma_base
        ruido = rng.normal(loc=0.0, scale=1.0, size=n)
        u_trial = u - eta*grad + sigma_actual*ruido
        p_trial = proj_cone(G, u_trial)
        norm_p = float(np.linalg.norm(p_trial))
        if norm_p < 1e-12:
            p_trial = proj_cone(G, u + sigma_actual*ruido)
            norm_p = float(np.linalg.norm(p_trial))
        if norm_p < 1e-12:
            historial_objetivo.append(F)
            historial_ruido.append(sigma_actual)
            historial_cambios.append(0.0)
            sin_mejora += 1
            continue
        u_new = p_trial/norm_p
        proy_new = proj_cone(H, u_new)
        F_new = 0.5 * float(np.dot(proy_new, proy_new))
        cos_new, v_new, polar_new = _soporte_y_direccion(
            H, u_new, proy_new, tol_polar)
        delta = float(np.linalg.norm(u_new-u))
        historial_objetivo.append(F_new)
        historial_ruido.append(sigma_actual)
        historial_cambios.append(delta)
        # Normalmente minimizar F equivale a minimizar cos(Theta) para cos>=0.
        # Dentro del polar todos empatan F=0: desempatar por coseno firmado.
        if F_new < F_best-1e-12 or (abs(F_new-F_best) <= 1e-12
                                    and cos_new < cos_best-1e-12):
            F_best = F_new
            cos_best = cos_new
            u_best = u_new.copy()
            v_best = v_new.copy()
            polar_best = polar_new
        if F_new < F-1e-12:
            sin_mejora = 0
        else:
            sin_mejora += 1
        u, proy, F = u_new, proy_new, F_new
        if es_polar(u, H, tol=tol_polar):
            converged = True
            motivo_parada = ('Se encontro direccion polar (F=0; '
                             'Theta no necesariamente optimo).')
            break

    # Conservar ANGULO numérico incluso si Proj_Q(u_best)=0 (caso polar).
    cos_best, v_best, polar_best = _soporte_y_direccion(H, u_best,
                                                       tol_polar=tol_polar)
    return {
        'u_opt': u_best, 'u_final': u, 'v_opt': v_best,
        'objetivo_final': F_best,
        'coseno': cos_best,
        'angulo': float(np.arccos(np.clip(cos_best, -1., 1.))),
        'es_polar': polar_best, 'polar': polar_best,  # alias de compatibilidad
        'tiempo_final': time.perf_counter()-inicio,
        'k': k_final, 'converged': converged,
        'motivo_parada': motivo_parada,
        'historial_objetivo': historial_objetivo,
        'historial_ruido': historial_ruido,
        'historial_cambios': historial_cambios,
    }


def _multistart_ruido(G, H, random_state=None, n_starts=EXCESS_N_STARTS):
    """Ejecuta el algoritmo ruidoso con arranques alternativos opcionales."""
    n_starts = max(1, int(n_starts))
    G_n = normalize_generators(G)
    H_n = normalize_generators(H)
    opciones = [None]
    if n_starts > 1:
        orden = sorted(range(G_n.shape[1]),
                       key=lambda j: _soporte_y_direccion(H_n, G_n[:, j])[0])
        opciones += [G_n[:, j] for j in orden]
    if random_state is None:
        seeds = [None] * n_starts
    else:
        spawned = np.random.SeedSequence(int(random_state)).spawn(n_starts)
        seeds = [int(s.generate_state(1)[0]) for s in spawned]
    best = None
    for ix in range(n_starts):
        ini = opciones[ix % len(opciones)]
        result = exceso_gradiente_ruido(
            G, H, u0=ini, eta=EXCESS_ETA, sigma0=EXCESS_SIGMA0,
            decay_ruido=EXCESS_DECAY_RUIDO, max_iter=EXCESS_MAX_ITER,
            eps=EXCESS_EPS, ventana=EXCESS_VENTANA,
            factor_escape=EXCESS_FACTOR_ESCAPE,
            random_state=seeds[ix], tol_polar=EXCESS_TOL_POLAR)
        if best is None or result['coseno'] < best['coseno']:
            best = result
    best['n_starts'] = n_starts
    return best


def distancia_conos_exceso_ruido(G, H, direction=EXCESS_DIRECTION, r=EXCESS_R,
                            random_state=None):
    """Mantiene el contrato del experimento: (score, dict_diagnostico)."""
    if direction not in ('directed', 'symmetric'):
        raise ValueError('direction debe ser directed o symmetric.')
    fw = _multistart_ruido(G, H, random_state=random_state)
    if direction == 'directed':
        return fw['angulo'], {
            'theta_PQ': fw['angulo'], 'theta_QP': np.nan,
            'cos_PQ': fw['coseno'], 'cos_QP': np.nan,
            'runtime_s': fw['tiempo_final'],
            'polar_PQ': fw['es_polar'], 'polar_QP': None,
            'conv_PQ': fw['converged'], 'conv_QP': None,
            'iters_PQ': fw['k'], 'iters_QP': np.nan,
            'objetivo_PQ': fw['objetivo_final'], 'objetivo_QP': np.nan,
            'n_starts': fw['n_starts'], 'sigma0': EXCESS_SIGMA0}
    seed_rev = None if random_state is None else (int(random_state)+1) % (2**32)
    rev = _multistart_ruido(H, G, random_state=seed_rev)
    v = np.array([np.sin(fw['angulo']/2), np.sin(rev['angulo']/2)])
    d = 2.0 * float(np.linalg.norm(v, ord=r))
    return d, {
        'theta_PQ': fw['angulo'], 'theta_QP': rev['angulo'],
        'cos_PQ': fw['coseno'], 'cos_QP': rev['coseno'],
        'runtime_s': fw['tiempo_final'] + rev['tiempo_final'],
        'polar_PQ': fw['es_polar'], 'polar_QP': rev['es_polar'],
        'conv_PQ': fw['converged'], 'conv_QP': rev['converged'],
        'iters_PQ': fw['k'], 'iters_QP': rev['k'],
        'objetivo_PQ': fw['objetivo_final'], 'objetivo_QP': rev['objetivo_final'],
        'n_starts': fw['n_starts'], 'sigma0': EXCESS_SIGMA0}


def cone_generators_for_excess(representation):
    # representation = (TF-MSM, conos_NMF_Sogi, embeddings_docs)
    if EXCESS_CONE_SOURCE == 'raw_docs':
        return representation[2]
    if EXCESS_CONE_SOURCE == 'sogi_nmf':
        return representation[1]
    raise ValueError('EXCESS_CONE_SOURCE debe ser raw_docs o sogi_nmf.')


In [9]:
def sanity_checks():
    a = np.array([[1., 0., 0., 0.]], dtype=np.float32)
    b = np.array([[0., 1., 0., 0.]], dtype=np.float32)
    fa = np.array([1.], dtype=np.float32)
    Ua, Ub = tf_subspace(a, fa, rank=1), tf_subspace(b, fa, rank=1)
    Ba, Bb = cone_basis(a, fa, rank=1), cone_basis(b, fa, rank=1)
    assert np.isclose(tf_msm_similarity(Ua, Ua, 1), 1., atol=1e-5)
    assert np.isclose(tf_msm_similarity(Ua, Ub, 1), 0., atol=1e-5)
    assert np.isclose(mcm_similarity(Ba, Ba, 1), 1., atol=1e-5)
    assert np.isclose(mcm_similarity(Ba, Bb, 1), 0., atol=1e-5)
    print('Verificaciones geometricas basicas: OK')

sanity_checks()


Verificaciones geometricas basicas: OK


In [10]:
def sanity_checks_exceso():
    e1 = np.array([[1.], [0.]])
    e2 = np.array([[0.], [1.]])
    neg_e1 = -e1
    ray_self = exceso_gradiente_proyectado(e1, e1)
    orth = exceso_gradiente_proyectado(e1, e2)
    polar = exceso_gradiente_proyectado(e1, neg_e1)
    assert np.isclose(ray_self['angulo'], 0., atol=1e-5)
    assert np.isclose(orth['angulo'], np.pi/2, atol=1e-5)
    assert np.isclose(polar['angulo'], np.pi, atol=1e-5), polar
    # Conos anidados: Theta(P,Q)=0 si P subconjunto Q;
    # la direccion inversa es distinta.
    P = e1
    Q = np.eye(2)
    d1, _ = distancia_conos_exceso(P, Q, direction='directed')
    d2, _ = distancia_conos_exceso(Q, P, direction='directed')
    assert np.isclose(d1, 0., atol=1e-5)
    assert np.isclose(d2, np.pi/2, atol=1e-5)
    ds, _ = distancia_conos_exceso(P, Q, direction='symmetric', r=2)
    assert np.isclose(ds, np.sqrt(2), atol=1e-5)
    print('Verificaciones del exceso angular (iguales/ortogonales/polares/anidados): OK')

sanity_checks_exceso()


Verificaciones del exceso angular (iguales/ortogonales/polares/anidados): OK


## 7. Clasificación de conjuntos y resultados

Para cada conjunto de M documentos, **Shimomoto** calcula una similitud TF-MSM por documento, por clase, y promedia los M puntajes antes del argmax. **Sogi y exceso** usan una matriz de M vectores documentales como cono de consulta. Se comparten conjuntos, etiquetas y particiones, pero no se obliga a compartir una representación inapropiada entre algoritmos. Se registran predicciones, accuracy por episodio, macro-F1, tiempos y excesos angulares por clase.


In [11]:
def document_matrix(docs, embeddings):
    # Filas = documentos, columnas = dimensiones del embedding.
    # Filtra documentos sin cobertura; nunca utiliza las etiquetas.
    X = np.stack([embeddings[ex['id']]['document'] for ex in docs]).astype(np.float32)
    X = X[np.linalg.norm(X, axis=1) > 1e-10]
    return normalize_rows(X)


def build_representations(docs, embeddings, as_query=False):
    """Shimomoto: palabras + TF; Sogi/exceso: embeddings completos de documentos.

    as_query=False: un subespacio TF-MSM de clase sobre el corpus de K textos.
    as_query=True: un subespacio TF-MSM distinto por cada texto de consulta;
                   los otros metodos usan un cono de M documentos.
    """
    if as_query:
        U = [tf_subspace(*aggregate_tokens([doc], embeddings, MAX_QUERY_TERMS))
             for doc in docs]
    else:
        U = tf_subspace(*aggregate_tokens(docs, embeddings, MAX_CLASS_TERMS))
    X = document_matrix(docs, embeddings)
    weights = np.ones(len(X), dtype=np.float32)
    B = cone_basis(X, weights)
    T = normalize_generators(X.T)
    return U, B, T


def evaluate_episode_at_k(ep, k, embeddings, query_repr=None):
    class_repr = {c: build_representations(ep['support'][c][:k], embeddings)
                  for c in ep['classes']}
    if query_repr is None:
        query_repr = [build_representations(q['docs'], embeddings, as_query=True)
                      for q in ep['queries']]
    y_true = [q['label'] for q in ep['queries']]
    method_names = ['Shimomoto_TF_MSM', 'Sogi_MCM', EXCESS_METHOD_NAME]
    predictions = {name: [] for name in method_names}
    runtimes = {name: 0. for name in method_names}
    details, scores_by_class = [], []

    for q, repq in zip(ep['queries'], query_repr):
        scores_t, scores_c, scores_e, info_e = {}, {}, {}, {}
        tic = time.perf_counter()
        for c in ep['classes']:
            scores_t[c] = float(np.mean([
                tf_msm_similarity(class_repr[c][0], Uq) for Uq in repq[0]
            ]))
        runtimes['Shimomoto_TF_MSM'] += time.perf_counter() - tic
        tic = time.perf_counter()
        for c in ep['classes']:
            scores_c[c] = mcm_similarity(class_repr[c][1], repq[1])
        runtimes['Sogi_MCM'] += time.perf_counter() - tic
        tic = time.perf_counter()
        G = cone_generators_for_excess(repq)
        for c in ep['classes']:
            H = cone_generators_for_excess(class_repr[c])
            if not G.shape[1] or not H.shape[1]:
                scores_e[c] = np.inf
                info_e[c] = {}
            else:
              clave = f"{SEED}|{ep['episode']}|{k}|{q['id']}|{c}"
              seed_pair = int.from_bytes(
                    hashlib.blake2b(clave.encode('utf-8'), digest_size=4).digest(),
                    byteorder='big')
              scores_e[c], info_e[c] = distancia_conos_exceso_ruido(
                    G, H, EXCESS_DIRECTION, EXCESS_R, random_state=seed_pair)
              # scores_e[c], info_e[c] = distancia_conos_exceso(
              #       G, H, EXCESS_DIRECTION, EXCESS_R)
        runtimes[EXCESS_METHOD_NAME] += time.perf_counter() - tic
        preds = {
            'Shimomoto_TF_MSM': max(ep['classes'], key=lambda c: scores_t[c]),
            'Sogi_MCM': max(ep['classes'], key=lambda c: scores_c[c]),
            EXCESS_METHOD_NAME: min(ep['classes'], key=lambda c: scores_e[c]),
        }
        for method, pred in preds.items():
            predictions[method].append(pred)
            sc = (scores_t if method == 'Shimomoto_TF_MSM' else
                  scores_c if method == 'Sogi_MCM' else scores_e)[pred]
            details.append({'query_id': q['id'], 'query_doc_ids': json.dumps(
                                [doc['id'] for doc in q['docs']]),
                            'M_query_docs': len(q['docs']), 'y_true': q['label'],
                            'y_pred': pred, 'method': method, 'score_best': sc})
        for c in ep['classes']:
            info = info_e[c]
            scores_by_class.append({'query_id': q['id'], 'M_query_docs': len(q['docs']),
                                    'y_true': q['label'], 'candidate_class': c,
                                    'score_excess': scores_e[c],
                                    'theta_PQ': info.get('theta_PQ', np.nan),
                                    'theta_QP': info.get('theta_QP', np.nan),
                                    'cos_PQ': info.get('cos_PQ', np.nan),
                                    'cos_QP': info.get('cos_QP', np.nan),
                                    'polar_PQ': info.get('polar_PQ'),
                                    'polar_QP': info.get('polar_QP'),
                                    'conv_PQ': info.get('conv_PQ'),
                                    'conv_QP': info.get('conv_QP'),
                                    'runtime_s': info.get('runtime_s', np.nan)})
    rows = []
    for method, y_pred in predictions.items():
        rows.append({'episode': ep['episode'], 'K_shot': k,
                     'N_way': len(ep['classes']), 'M_query_docs': M_QUERY_DOCS,
                     'method': method, 'accuracy': accuracy_score(y_true, y_pred),
                     'macro_f1': f1_score(y_true, y_pred, labels=ep['classes'],
                                          average='macro', zero_division=0),
                     'n_query_sets': len(y_true), 'elapsed_method_s': runtimes[method],
                     'class_ids': json.dumps(ep['classes'])})
    return rows, details, scores_by_class


def save_results(rows, predictions, scores):
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / 'resultados_por_episodio.csv', index=False)
    pd.DataFrame(predictions).to_csv(OUTPUT_DIR / 'predicciones.csv', index=False)
    pd.DataFrame(scores).to_csv(OUTPUT_DIR / 'exceso_por_clase.csv', index=False)
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return None
    group = ['dataset', 'embedding', 'method', 'N_way', 'K_shot', 'M_query_docs']
    summary = df.groupby(group, as_index=False).agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        n_episodes=('accuracy', 'count'),
        avg_runtime_s=('elapsed_method_s', 'mean'))
    summary[['accuracy_std', 'macro_f1_std']] = summary[
        ['accuracy_std', 'macro_f1_std']].fillna(0.)
    summary.to_csv(OUTPUT_DIR / 'resultados_resumen_K1_K30.csv', index=False)
    return summary


def plot_results(summary):
    for dataset in sorted(summary['dataset'].unique()):
        df = summary[summary['dataset'] == dataset]
        fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
        for (emb, method), d in df.groupby(['embedding', 'method']):
            d = d.sort_values('K_shot')
            name = emb.split('/')[-1]
            ax.errorbar(d['K_shot'], d['accuracy_mean'],
                        yerr=d['accuracy_std'],
                        marker='.', ms=3, linewidth=1.4,
                        capsize=0, label=f'{name} | {method}')
        ax.set(title=f'{dataset} — Accuracy por K-shot (N-way)',
               xlabel='K (ejemplos por clase)', ylabel='Accuracy')
        ax.set_xticks(K_VALUES)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=.25)
        ax.legend(loc='best', fontsize=7)
        fig.savefig(OUTPUT_DIR / f'accuracy_{dataset}.png', dpi=160)
        plt.close(fig)


def main():
    task_data = {}
    for spec in DATASETS:
        print(f'Preparando: {spec.name}')
        pool = make_pools(spec)
        episodes = create_episodes(spec, pool)
        task_data[spec.name] = (pool, episodes)
    config_path = OUTPUT_DIR / 'config_experimento.json'
    config = {'seed': SEED, 'k_values': K_VALUES, 'n_way_target': N_WAY,
              'episodes': N_EPISODES, 'query_sets_per_class': N_QUERY, 'm_query_docs': M_QUERY_DOCS,
              'embedding_names': EMBEDDINGS, 'n_angles': N_ANGLES,
              'representation': 'shimomoto_token_tf_msm_per_query_doc_mean_scores__sogi_excess_document_cones',
              'tf_msm_max_class_terms': MAX_CLASS_TERMS,
              'tf_msm_max_query_terms': MAX_QUERY_TERMS,
              'subspace_rank': SUBSPACE_RANK, 'cone_rank': CONE_RANK,
              'max_words_doc': DOC_MAX_WORDS,
              'transformer_max_pieces': TRANSFORMER_MAX_PIECES,
              'excess_direction': EXCESS_DIRECTION, 'excess_source': EXCESS_CONE_SOURCE,
              'excess_r': str(EXCESS_R), 'excess_max_iter': EXCESS_MAX_ITER,
              'excess_starts': EXCESS_N_STARTS,
              'datasets': [vars(s) for s in DATASETS]}
    if config_path.exists():
        with open(config_path, encoding='utf8') as fh:
            previous = json.load(fh)
        if previous != config:
            raise RuntimeError('Hay resultados con otra configuracion en OUTPUT_DIR. '
                               'Usa otro directorio o borra los archivos previos.')
    with open(config_path, 'w', encoding='utf8') as fh:
        json.dump(config, fh, indent=2)

    episodes_csv = OUTPUT_DIR / 'resultados_por_episodio.csv'
    predictions_csv = OUTPUT_DIR / 'predicciones.csv'
    scores_csv = OUTPUT_DIR / 'exceso_por_clase.csv'
    rows = pd.read_csv(episodes_csv).to_dict('records') if episodes_csv.exists() else []
    predictions = pd.read_csv(predictions_csv).to_dict('records') if predictions_csv.exists() else []
    scores = pd.read_csv(scores_csv).to_dict('records') if scores_csv.exists() else []
    completed = {(r['dataset'], r['embedding'], int(r['episode']), int(r['K_shot']))
                 for r in rows if r['method'] == EXCESS_METHOD_NAME}
    for embed_name in EMBEDDINGS:
        print('\n' + '=' * 70 + f'\nEMBEDDING {embed_name}')
        pending = [(spec.name, ep) for spec in DATASETS for ep in range(N_EPISODES)
                   for k in K_VALUES if (spec.name, embed_name, ep, k) not in completed]
        if not pending:
            print('Completado previamente; se omite el modelo.')
            continue
        encoder = instantiate_encoder(embed_name)
        for spec in DATASETS:
            print(f'\nDataset {spec.name}')
            if all((spec.name, embed_name, ep, k) in completed
                   for ep in range(N_EPISODES) for k in K_VALUES):
                print(' Dataset ya procesado; se omite.')
                continue
            pool, episodes = task_data[spec.name]
            embedded = encode_pool(encoder, pool)
            for ep in tqdm(episodes, desc=f'{spec.name}: episodios'):
                query_repr = [build_representations(q['docs'], embedded, as_query=True)
                              for q in ep['queries']]
                for k in tqdm(K_VALUES, desc=f'K de episodio {ep["episode"]}', leave=False):
                    item_key = (spec.name, embed_name, ep['episode'], k)
                    if item_key in completed:
                        continue
                    batch_rows, batch_details, batch_scores = evaluate_episode_at_k(
                        ep, k, embedded, query_repr)
                    for r in batch_rows:
                        rows.append({'dataset': spec.name, 'embedding': embed_name, **r})
                    completed.add(item_key)
                    for d in batch_details:
                        predictions.append({'dataset': spec.name,
                                            'embedding': embed_name,
                                            'episode': ep['episode'],
                                            'N_way': len(ep['classes']), 'M_query_docs': M_QUERY_DOCS,
                                            'K_shot': k, **d})
                    for d in batch_scores:
                        scores.append({'dataset': spec.name, 'embedding': embed_name,
                                       'episode': ep['episode'],
                                       'N_way': len(ep['classes']), 'M_query_docs': M_QUERY_DOCS,
                                       'K_shot': k, **d})
                summary = save_results(rows, predictions, scores)
            del embedded
            gc.collect()
        del encoder
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    summary = save_results(rows, predictions, scores)
    plot_results(summary)
    print('\nArchivos:', OUTPUT_DIR.resolve())
    print(summary.head(12).to_string(index=False))
    return summary


## 8. Ejecutar

Se descargan las bases y embeddings la primera vez. El número de documentos de consulta `M_QUERY_DOCS` aparece en configuración, CSVs y nombres de grupos. En la primera evaluación, mantener `PILOTO=True` y luego pasar a `False`; ejecutar todos los experimentos puede requerir considerable tiempo en Colab gratuito.


In [ ]:
if __name__ == '__main__':
    resumen = main()
    import shutil
    shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
    print('ZIP:', str(OUTPUT_DIR) + '.zip')
    # En Colab, para bajar el ZIP:
    # from google.colab import files
    # files.download(str(OUTPUT_DIR) + '.zip')


Preparando: AG_News


README.md: 0.00B [00:00, ?B/s]

C:\Users\pedro\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pedro\.cache\huggingface\hub\datasets--fancyzhx--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


 AG_News: 4 clases aptas, train/test independientes
Preparando: DBpedia_14


README.md: 0.00B [00:00, ?B/s]

C:\Users\pedro\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pedro\.cache\huggingface\hub\datasets--fancyzhx--dbpedia_14. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


... (67 outputs left out)

K de episodio 0:   0%|          | 0/20 [00:00<?, ?it/s]

K de episodio 1:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

: 

In [15]:
"""Tablas y curvas de aprendizaje few-shot para Shimomoto, Sogi y exceso angular.
Ejecutar en Colab tras finalizar el notebook principal, o localmente con sus CSV.
"""
# %%writefile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# En Colab, coincide con OUTPUT_DIR del notebook principal.
RESULTS_DIR = Path('/content/resultados_tres_metodos_1')
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path('./resultados_tres_metodos')
OUT_DIR = RESULTS_DIR / 'analisis_comparativo'
OUT_DIR.mkdir(parents=True, exist_ok=True)

METODOS = {
    'Shimomoto_TF_MSM': 'Shimomoto (TF-MSM)',
    'Sogi_MCM': 'Sogi (MCM)',
    'Oliveira_Exceso_Dirigido': 'Exceso angular (dirigido)',
    'Oliveira_Exceso_Simetrico': 'Exceso angular (simétrico)',
}
METRICAS = ['accuracy', 'macro_f1']
K_REFERENCIA = [1, 5, 10, 20, 30]


def cargar_resultados(carpeta=RESULTS_DIR):
    """Prefiere los datos por episodio: permiten calcular dispersión sin redondeos."""
    carpeta = Path(carpeta)
    episodios = carpeta / 'resultados_por_episodio.csv'
    resumen = carpeta / 'resultados_resumen_K1_K30.csv'
    if episodios.exists():
        df = pd.read_csv(episodios)
        obligatorias = {'dataset', 'embedding', 'method', 'K_shot', 'accuracy', 'macro_f1'}
        faltan = obligatorias - set(df.columns)
        if faltan:
            raise ValueError(f'Faltan columnas en {episodios.name}: {faltan}')
        df = df.dropna(subset=list(obligatorias)).copy()
        if 'M_query_docs' not in df:
            df['M_query_docs'] = 5
        if 'N_way' not in df:
            df['N_way'] = np.nan
        keys = ['dataset', 'embedding', 'method', 'N_way', 'M_query_docs', 'K_shot']
        agg = {}
        for metrica in METRICAS:
            agg[f'{metrica}_mean'] = (metrica, 'mean')
            agg[f'{metrica}_std'] = (metrica, 'std')
        agg['n_episodes'] = ('accuracy', 'size')
        if 'elapsed_method_s' in df:
            agg['avg_runtime_s'] = ('elapsed_method_s', 'mean')
        result = df.groupby(keys, as_index=False, dropna=False).agg(**agg)
        for metrica in METRICAS:
            result[f'{metrica}_std'] = result[f'{metrica}_std'].fillna(0.0)
        return result
    if resumen.exists():
        result = pd.read_csv(resumen)
        if 'M_query_docs' not in result:
            result['M_query_docs'] = 5
        if 'N_way' not in result:
            result['N_way'] = np.nan
        return result
    raise FileNotFoundError(
        f'No hay CSV en {carpeta}. Ejecuta primero el notebook experimental.'
    )


def etiqueta_metodo(nombre):
    return METODOS.get(nombre, nombre)


def tablas_comparativas(df, carpeta=OUT_DIR):
    """Exporta comparación por K y tabla resumen (sin perder K intermedios)."""
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    detalle = df.copy()
    detalle['metodo'] = detalle['method'].map(etiqueta_metodo)
    detalle.to_csv(carpeta / 'tabla_detallada_todos_los_K.csv', index=False)

    indices = ['dataset', 'embedding', 'N_way', 'M_query_docs', 'K_shot']
    for metrica in METRICAS:
        tabla = detalle.pivot_table(
            index=indices,
            columns='metodo',
            values=f'{metrica}_mean',
            aggfunc='first',
        )
        tabla = (100 * tabla).round(2)
        tabla.to_csv(carpeta / f'tabla_{metrica}_por_K_pct.csv')

    filas = []
    grupos = ['dataset', 'embedding', 'N_way', 'M_query_docs', 'method']
    for valores, g in detalle.groupby(grupos, dropna=False):
        g = g.sort_values('K_shot').drop_duplicates('K_shot')
        fila = dict(zip(grupos, valores))
        fila['metodo'] = etiqueta_metodo(fila['method'])
        fila['K_evaluados'] = len(g)
        fila['K_min'] = int(g['K_shot'].min())
        fila['K_max'] = int(g['K_shot'].max())
        fila['episodios_min'] = int(g['n_episodes'].min()) if 'n_episodes' in g else np.nan
        for metrica in METRICAS:
            datos = g.set_index('K_shot')[f'{metrica}_mean']
            fila[f'{metrica}_promedio_K_pct'] = 100 * datos.mean()
            fila[f'{metrica}_AULC_norm_pct'] = (
                100 * np.trapezoid(datos.to_numpy(), g['K_shot'].to_numpy()) /
                (g['K_shot'].max() - g['K_shot'].min())
                if len(g) > 1 else np.nan
            )
            fila[f'{metrica}_mejor_K'] = int(datos.idxmax())
            fila[f'{metrica}_mejor_pct'] = 100 * datos.max()
            for k in K_REFERENCIA:
                fila[f'{metrica}_K{k}_pct'] = 100 * datos.loc[k] if k in datos.index else np.nan
        if 'avg_runtime_s' in g:
            fila['tiempo_medio_episodio_s'] = g['avg_runtime_s'].mean()
        filas.append(fila)
    resumen = pd.DataFrame(filas)
    # Una puntuación global solo es comparable si hay exactamente los mismos K evaluados.
    resumen = resumen.sort_values(['dataset', 'embedding', 'metodo'])
    resumen.to_csv(carpeta / 'tabla_resumen_comparativa.csv', index=False)
    return resumen


def graficar_curvas(df, carpeta=OUT_DIR, rango_k=(1, 30)):
    """Genera 2 métricas x cada dataset x embedding, tres curvas por figura."""
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    df = df[(df['K_shot'] >= rango_k[0]) & (df['K_shot'] <= rango_k[1])]
    for (dataset, embedding, n_way, m), grupo in df.groupby(
        ['dataset', 'embedding', 'N_way', 'M_query_docs'], dropna=False
    ):
        for metrica in METRICAS:
            fig, ax = plt.subplots(figsize=(11, 5.7))
            for metodo, d in grupo.groupby('method'):
                d = d.sort_values('K_shot')
                x = d['K_shot'].to_numpy(dtype=int)
                y = d[f'{metrica}_mean'].to_numpy(dtype=float) * 100
                sd = d.get(f'{metrica}_std', pd.Series(0.0, index=d.index)).to_numpy(dtype=float) * 100
                linea, = ax.plot(x, y, marker='o', markersize=3.2,
                                 linewidth=1.8, label=etiqueta_metodo(metodo))
                ax.fill_between(x, np.maximum(0, y-sd), np.minimum(100, y+sd),
                                color=linea.get_color(), alpha=0.12)
            ax.set(title=f'{dataset} | {embedding.split("/")[-1]} | {n_way}-way | m={m}',
                   xlabel='K-shot (documentos de entrenamiento por clase)',
                   ylabel=f'{metrica.replace("_", " ").upper()} (%)')
            ax.set_xticks(sorted(grupo['K_shot'].unique()))
            ax.set_ylim(0, 102)
            ax.grid(True, alpha=0.25)
            ax.legend(loc='best', fontsize=9)
            fig.tight_layout()
            safe = lambda s: ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in str(s))
            nombre = f'{safe(dataset)}_{safe(embedding.split("/")[-1])}_N{safe(n_way)}_m{safe(m)}_{metrica}.png'
            fig.savefig(carpeta / nombre, dpi=180, bbox_inches='tight')
            plt.close(fig)


def graficar_promedio_datasets(df, carpeta=OUT_DIR):
    """Macro-promedia datasets por K, dentro de cada embedding y método.

    Úsalo solo como visualización complementaria: los datasets tienen distintas tareas
    y algunos tienen diferente número de clases N-way.
    """
    for embedding, e in df.groupby('embedding'):
        # Promediar primero en cada dataset si hay distintos valores de N o m.
        promedio = (e.groupby(['dataset', 'method', 'K_shot'], as_index=False)
                      ['accuracy_mean'].mean())
        fig, ax = plt.subplots(figsize=(11, 5.6))
        for metodo, g in promedio.groupby('method'):
            curva = (g.groupby('K_shot')['accuracy_mean']
                       .agg(['mean', 'std']).reset_index().sort_values('K_shot'))
            x = curva['K_shot'].to_numpy()
            y = 100 * curva['mean'].to_numpy()
            sd = 100 * curva['std'].fillna(0).to_numpy()
            linea, = ax.plot(x, y, marker='o', ms=3, lw=1.8,
                             label=etiqueta_metodo(metodo))
            ax.fill_between(x, np.maximum(0, y-sd), np.minimum(100, y+sd),
                            alpha=.12, color=linea.get_color())
        ax.set(xlabel='K-shot', ylabel='Accuracy promedio entre datasets (%)',
               title=f'Promedio entre datasets | {embedding.split("/")[-1]}')
        ax.set_ylim(0, 102)
        ax.set_xticks(sorted(e['K_shot'].unique()))
        ax.grid(alpha=.25)
        ax.legend()
        fig.tight_layout()
        archivo = ''.join(c if c.isalnum() or c in '_-' else '_' for c in embedding.split('/')[-1])
        fig.savefig(Path(carpeta) / f'promedio_datasets_{archivo}.png', dpi=180)
        plt.close(fig)


def main():
    resultados = cargar_resultados()
    tabla = tablas_comparativas(resultados)
    graficar_curvas(resultados)
    graficar_promedio_datasets(resultados)
    columnas = ['dataset', 'embedding', 'metodo', 'K_evaluados',
                'accuracy_promedio_K_pct', 'accuracy_K1_pct', 'accuracy_K10_pct',
                'accuracy_K30_pct', 'accuracy_mejor_K']
    print('\nTABLA RESUMEN (porcentaje, K faltantes = NaN):')
    print(tabla[columnas].round(2).to_string(index=False))
    print(f'\nArchivos guardados en: {OUT_DIR.resolve()}')


if __name__ == '__main__':
    main()



TABLA RESUMEN (porcentaje, K faltantes = NaN):
      dataset                              embedding              metodo  K_evaluados  accuracy_promedio_K_pct  accuracy_K1_pct  accuracy_K10_pct  accuracy_K30_pct  accuracy_mejor_K
20_Newsgroups                 BAAI/bge-small-en-v1.5 Distancia Simétrica           20                    73.03            56.67             69.33               NaN                19
20_Newsgroups                 BAAI/bge-small-en-v1.5  Shimomoto (TF-MSM)           20                    78.83            64.00             80.67               NaN                12
20_Newsgroups                 BAAI/bge-small-en-v1.5          Sogi (MCM)           20                    87.07            62.00             90.67               NaN                18
20_Newsgroups sentence-transformers/all-MiniLM-L6-v2 Distancia Simétrica           20                    74.70            56.67             72.67               NaN                20
20_Newsgroups sentence-transformers/all-Mi

In [16]:
from google.colab import files
files.download(str(OUTPUT_DIR) + '.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>